read sample csv, download images from url column into data/images, and save updated csv with local_path column

In [9]:
import os
import time
import pandas as pd
import requests
from tqdm import tqdm
from urllib.parse import urlparse

SAMPLE_CSV = "/Users/mariaworkman/VSCode25/fashion-neutrality/data/samples/sample_2010_2023_100peryear.csv"
OUT_DIR = "../data/images/sample_2010_2023_100peryear"
OUT_CSV = "../data/samples/sample_2010_2023_100peryear_with_paths.csv"

os.makedirs(OUT_DIR, exist_ok=True)

In [10]:
df = pd.read_csv(SAMPLE_CSV)
df.shape, df.columns
df[["year", "url", "filename"]].head()

,year,url,filename
0,2010,https://assets.vogue.com/photos/55c651b808298d...,0851646_Jen Kao - Fall 2010 Ready-to-Wear [Col...
1,2010,https://assets.vogue.com/photos/55c650ab08298d...,0144146_Jean Paul Gaultier - Fall 2010 Menswea...
2,2010,https://assets.vogue.com/photos/55c651bd08298d...,0518215_Aquilano.Rimondi - Fall 2010 Ready-to-...
3,2010,https://assets.vogue.com/photos/55c651af08298d...,1013544_Talbot Runhof - Spring 2010 Ready-to-W...
4,2010,https://assets.vogue.com/photos/55c651a908298d...,0286185_Nicole Farhi - Spring 2010 Ready-to-We...


In [11]:
def safe_filename_from_row(row, i):
    """
    Prefer dataset filename; otherwise make a stable name from key/year/index.
    """
    if "filename" in row and isinstance(row["filename"], str) and row["filename"].strip():
        # strip weird path separators just in case
        return row["filename"].replace("/", "_")
    if "key" in row:
        return f"{int(row['key']):07d}_y{int(row['year'])}.jpg"
    return f"img_{i:06d}_y{int(row['year'])}.jpg"

def download_with_retries(url, out_path, retries=3, timeout=30, sleep=1.0):
    """
    Downloads url to out_path with a few retries.
    Returns True if saved, False otherwise.
    """
    headers = {"User-Agent": "Mozilla/5.0"}  # helps some CDNs

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            r.raise_for_status()

            # Basic content-type guard (optional)
            ctype = r.headers.get("Content-Type", "")
            if "image" not in ctype:
                # sometimes CDNs return HTML error pages
                raise ValueError(f"Non-image content-type: {ctype}")

            with open(out_path, "wb") as f:
                f.write(r.content)
            return True

        except Exception as e:
            if attempt == retries:
                return False
            time.sleep(sleep * attempt)  # simple backoff


In [12]:
df = df.copy()
df["local_path"] = None
df["download_ok"] = False

TEST_N = 20
test_df = df.head(TEST_N)

In [13]:
for i, row in tqdm(list(test_df.iterrows()), total=len(test_df)):
    url = row["url"]
    fname = safe_filename_from_row(row, i)

    # keep extension if present in filename, else default .jpg
    if "." not in fname:
        fname += ".jpg"

    out_path = os.path.join(OUT_DIR, fname)

    # skip if already exists
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        df.at[i, "local_path"] = out_path
        df.at[i, "download_ok"] = True
        continue

    ok = download_with_retries(url, out_path, retries=3)
    df.at[i, "local_path"] = out_path if ok else None
    df.at[i, "download_ok"] = ok


100%|██████████| 20/20 [00:21<00:00,  1.08s/it]


In [14]:
df.head(TEST_N)[["year", "download_ok", "local_path"]]
df["download_ok"].head(TEST_N).mean()

np.float64(1.0)

In [17]:
for i, row in tqdm(list(df.iterrows()), total=len(df)):
    if df.at[i, "download_ok"] is True and isinstance(df.at[i, "local_path"], str):
        continue

    url = row["url"]
    fname = safe_filename_from_row(row, i)

    if "." not in fname:
        fname += ".jpg"

    out_path = os.path.join(OUT_DIR, fname)

    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        df.at[i, "local_path"] = out_path
        df.at[i, "download_ok"] = True
        continue

    ok = download_with_retries(url, out_path, retries=3)
    df.at[i, "local_path"] = out_path if ok else None
    df.at[i, "download_ok"] = ok


100%|██████████| 1400/1400 [41:42<00:00,  1.79s/it]   
